# Лабораторна робота №2 - Частина 2
## Дослідження Individual Household Electric Power Consumption Dataset

**Мета:** Використання `pandas`, `timeit` для обробки великих наборів даних, обчислення кореляцій та One Hot Encoding.

In [1]:
import pandas as pd
import numpy as np
import timeit


# Допоміжна функція для профілювання часу
def profile_execution(func, *args, **kwargs):
    start = timeit.default_timer()
    result = func(*args, **kwargs)
    end = timeit.default_timer()
    print(f"[{func.__name__}] Час виконання: {end - start:.5f} сек.")
    return result

### 1. Зчитування та Data Cleaning
Завантажуємо датасет, перетворюємо '?' на NaN, видаляємо пропуски, зводимо дату та час у зручний формат.

In [2]:
import urllib.request
import zipfile
import os

def download_and_extract_power_data():
    # Використовуємо твоє актуальне посилання
    url = "https://archive.ics.uci.edu/static/public/235/individual+household+electric+power+consumption.zip"
    zip_path = "individual+household+electric+power+consumption.zip"
    txt_path = "household_power_consumption.txt"

    if not os.path.exists(txt_path):
        print("Завантаження датасету (це може зайняти хвилину)...")
        urllib.request.urlretrieve(url, zip_path)
        print("Розпакування...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall()
        os.remove(zip_path) # Видаляємо архів, щоб не займав місце
        print("Готово!")
    else:
        print("Датасет вже існує локально.")

# Викликаємо автоматичне завантаження перед парсингом
download_and_extract_power_data()

Датасет вже існує локально.


In [3]:
def load_and_clean_data(filepath='household_power_consumption.txt'):
    # Зчитування з обробкою '?' як NaN
    df = pd.read_csv(filepath, sep=';', na_values=['?'], 
                     dtype={'Global_active_power': float, 'Global_reactive_power': float, 
                            'Voltage': float, 'Global_intensity': float, 
                            'Sub_metering_1': float, 'Sub_metering_2': float, 'Sub_metering_3': float})
    
    # Видалення рядків з NaN
    df = df.dropna()
    
    # Створення зручної колонки Datetime
    df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M:%S')
    
    return df

print("Зчитування даних (це може зайняти кілька секунд)...")
df_power = profile_execution(load_and_clean_data)
print("Дані завантажено успішно. Розмір:", df_power.shape)

Зчитування даних (це може зайняти кілька секунд)...
[load_and_clean_data] Час виконання: 7.03147 сек.
Дані завантажено успішно. Розмір: (2049280, 10)


### 2. Формування вибірок
Реалізація чотирьох специфічних запитів. Кожна функція обгорнута у профілювальник часу.

In [4]:
def query_1_high_power(df):
    return df[df['Global_active_power'] > 5.0]

res1 = profile_execution(query_1_high_power, df_power)
print(f"Знайдено записів (Потужність > 5): {len(res1)}")

[query_1_high_power] Час виконання: 0.00631 сек.
Знайдено записів (Потужність > 5): 17547


In [5]:
def query_2_intensity_and_metering(df):
    mask = (df['Global_intensity'] >= 19.0) & (df['Global_intensity'] <= 20.0) & \
           (df['Sub_metering_2'] > df['Sub_metering_3'])
    return df[mask]

res2 = profile_execution(query_2_intensity_and_metering, df_power)
print(f"Знайдено записів (Струм 19-20А, Sub2 > Sub3): {len(res2)}")

[query_2_intensity_and_metering] Час виконання: 0.01475 сек.
Знайдено записів (Струм 19-20А, Sub2 > Sub3): 2509


In [6]:
def query_3_random_sample_means(df):
    sample = df.sample(n=500000, replace=False, random_state=42)
    return sample[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()

res3 = profile_execution(query_3_random_sample_means, df_power)
print(f"Середні значення (Випадкові 500k):\n{res3.to_string()}")

[query_3_random_sample_means] Час виконання: 0.21743 сек.
Середні значення (Випадкові 500k):
Sub_metering_1    1.119258
Sub_metering_2    1.308912
Sub_metering_3    6.452950


In [7]:
def query_4_complex_evening_filter(df):
    filtered = df[(df['Datetime'].dt.hour >= 18) & (df['Global_active_power'] > 6.0)]
    mask_group2_biggest = (filtered['Sub_metering_2'] > filtered['Sub_metering_1']) & \
                          (filtered['Sub_metering_2'] > filtered['Sub_metering_3'])
    filtered = filtered[mask_group2_biggest]
    
    half_idx = len(filtered) // 2
    res1 = filtered.iloc[:half_idx].iloc[2::3]
    res2 = filtered.iloc[half_idx:].iloc[3::4]
    return pd.concat([res1, res2])

res4 = profile_execution(query_4_complex_evening_filter, df_power)
print(f"Знайдено записів (Вечірні складні фільтри): {len(res4)}")

[query_4_complex_evening_filter] Час виконання: 0.05681 сек.
Знайдено записів (Вечірні складні фільтри): 308


### 3. Нормування, Стандартизація та Кореляція

In [8]:
def normalize_and_standardize(df, column):
    normalized = (df[column] - df[column].min()) / (df[column].max() - df[column].min())
    standardized = (df[column] - df[column].mean()) / df[column].std()
    return normalized, standardized

norm, stand = profile_execution(normalize_and_standardize, df_power, 'Global_active_power')
print("Приклад нормалізації (перші 3):\n", norm.head(3).values)
print("Приклад стандартизації (перші 3):\n", stand.head(3).values)

[normalize_and_standardize] Час виконання: 0.05547 сек.
Приклад нормалізації (перші 3):
 [0.37479631 0.47836321 0.47963064]
Приклад стандартизації (перші 3):
 [2.95507634 4.03708364 4.05032499]


In [9]:
def calculate_correlations(df, col1, col2):
    pearson_corr = df[col1].corr(df[col2], method='pearson')
    spearman_corr = df[col1].corr(df[col2], method='spearman')
    return pearson_corr, spearman_corr

p_corr, s_corr = profile_execution(calculate_correlations, df_power, 'Global_active_power', 'Global_intensity')
print(f"\nКоефіцієнт Пірсона: {p_corr:.4f}")
print(f"Коефіцієнт Спірмена: {s_corr:.4f}")

[calculate_correlations] Час виконання: 1.57545 сек.

Коефіцієнт Пірсона: 0.9989
Коефіцієнт Спірмена: 0.9954


### 4. One Hot Encoding категоріального атрибута
Оскільки явно категоріальних текстових атрибутів немає, створимо колонку "День тижня" з Datetime та застосуємо One Hot Encoding до неї.

In [10]:
def do_one_hot_encoding(df):
    df_temp = df.copy()
    df_temp['DayOfWeek'] = df_temp['Datetime'].dt.day_name()
    return pd.get_dummies(df_temp, columns=['DayOfWeek'], dtype=int)

df_encoded = profile_execution(do_one_hot_encoding, df_power)
print("\nКолонки після One Hot Encoding:")
display(df_encoded.iloc[:, -8:].sample(5))

[do_one_hot_encoding] Час виконання: 0.62296 сек.

Колонки після One Hot Encoding:


,Datetime,DayOfWeek_Friday,DayOfWeek_Monday,DayOfWeek_Saturday,DayOfWeek_Sunday,DayOfWeek_Thursday,DayOfWeek_Tuesday,DayOfWeek_Wednesday
874588,2008-08-15 01:52:00,1,0,0,0,0,0,0
310187,2007-07-20 03:11:00,1,0,0,0,0,0,0
1668512,2010-02-17 09:56:00,0,0,0,0,0,0,1
1718160,2010-03-23 21:24:00,0,0,0,0,0,1,0
1832534,2010-06-11 07:38:00,1,0,0,0,0,0,0
